In [1]:
# ============================================================
# IMPORTS
# ============================================================
import os
import random
import copy
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from torchvision import transforms, models
from torchvision.models import ViT_B_16_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)
from sklearn.manifold import TSNE
from matplotlib.colors import ListedColormap
from tqdm import tqdm


# Optional: used for histogram matching in one preprocessing pipeline.
try:
    from skimage.exposure import match_histograms
except Exception:
    match_histograms = None


In [5]:
# ============================================================
# CONFIGURATION
# ============================================================
DATA_ROOT = Path("/kaggle/input/datasets/paulconrad21/eye-disease-dataset/eye-disease-data")

# CSV files
TRAIN_CSV_PATH = DATA_ROOT / "2. Groundtruths" / "a. IDRiD_Disease Grading_Training Labels.csv"
TEST_CSV_PATH  = DATA_ROOT / "2. Groundtruths" / "b. IDRiD_Disease Grading_Testing Labels.csv"

# Image folders
TRAIN_IMAGE_DIR = DATA_ROOT / "1. Original Images" / "a. Training Set"
TEST_IMAGE_DIR  = DATA_ROOT / "1. Original Images" / "b. Testing Set"
# ----------------------------
# Dataset columns
# ----------------------------

IMAGE_COL = "image"
LABEL_COL = "level"

NUM_CLASSES = 5

# ----------------------------
# Hardware / Device
# ----------------------------

# Options:
# "auto"  -> use GPU if available
# "cuda"  -> force GPU (will crash if not available)
# "cpu"   -> force CPU
DEVICE_MODE = "auto"

# ----------------------------
# Image settings
# ----------------------------

IMAGE_SIZE = 224  # 224, 384, 512

# ----------------------------
# Training settings
# ----------------------------

BATCH_SIZE = 16
NUM_WORKERS = 2  # Kaggle often works better with 2–4
NUM_EPOCHS = 20
PATIENCE = 8
SEED = 42

# ----------------------------
# Augmentation
# ----------------------------

USE_AUGMENTATION = False
USE_RANDOM_FLIP = False
USE_RANDOM_ROTATION = False
USE_COLOR_JITTER = False
USE_RANDOM_RESIZED_CROP = False

# ----------------------------
# Loss selection (ONLY ONE)
# ----------------------------

LOSS_NAME = "weighted_cross_entropy"

# Options:
# "cross_entropy"
# "weighted_cross_entropy"
# "focal_loss"
# "weighted_focal_loss"

FOCAL_GAMMA = 2.128858868487051

# ----------------------------
# Optimizer selection
# ----------------------------

OPTIMIZER_NAME = "adamw"

# Options:
# "adam"
# "adamw"
# "sgd"

LEARNING_RATE = 1.6356792619359833e-05
WEIGHT_DECAY = 0.0006515263398016582  # AdamW regularization
MOMENTUM = 0.9       # only used for SGD

# ----------------------------
# Sampling
# ----------------------------

USE_WEIGHTED_SAMPLER = False
# ----------------------------
# Validation split
# ----------------------------
VAL_SIZE = 0.2
USE_STRATIFIED_SPLIT = True
# ============================================================
# AUTO-DERIVED SETTINGS (prevents mistakes)
# ============================================================
USE_WEIGHTED_LOSS = LOSS_NAME in ["weighted_cross_entropy", "weighted_focal_loss"]


# ============================================================
# DEVICE SETUP
# ============================================================

def get_device():
    if DEVICE_MODE == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    elif DEVICE_MODE == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but not available.")
        return torch.device("cuda")
    elif DEVICE_MODE == "cpu":
        return torch.device("cpu")
    else:
        raise ValueError(f"Invalid DEVICE_MODE: {DEVICE_MODE}")

DEVICE = get_device()

# ============================================================
# Fixed best hyperparameters from trial 7
# Class-specific oversampling multipliers removed: training now uses the original train split only.
# ============================================================
DROPOUT = 0.08572287701812711
MLP_DEPTH = 1
HIDDEN_DIM = 512
DATA_AUGMENTATION_MULTIPLIER = 2
BEST_TRIAL_NUMBER = 7
BEST_VALIDATION_SCORE = 0.6817124625495482

PREPROCESSING_MODES = [
    "standard",
    "green_channel",
    "clahe",
    "gamma_correction",
    "gray_gamma_clahe",
    "ben_graham",
    "white_balance",
    "white_balance_clahe",
]


In [6]:
# ============================================================
# SEED (reproducibility)
# ============================================================
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

In [ ]:
# ============================================================
# CONFIG VALIDATION
# ============================================================
def validate_config():

    valid_losses = [
        "cross_entropy",
        "weighted_cross_entropy",
        "focal_loss",
        "weighted_focal_loss",
    ]

    if LOSS_NAME not in valid_losses:
        raise ValueError(f"Invalid LOSS_NAME: {LOSS_NAME}")

    # prevent conflicting imbalance strategies
    if USE_WEIGHTED_LOSS and USE_WEIGHTED_SAMPLER:
        raise ValueError(
            "Weighted loss and weighted sampler should not be enabled at the same time."
        )
    
    valid_optimizers = ["adam", "adamw", "sgd"]

    if OPTIMIZER_NAME not in valid_optimizers:
        raise ValueError(f"Invalid OPTIMIZER_NAME: {OPTIMIZER_NAME}")
    
    if IMAGE_SIZE <= 0:
        raise ValueError("IMAGE_SIZE must be > 0")

    if BATCH_SIZE <= 0:
        raise ValueError("BATCH_SIZE must be > 0")

    if NUM_CLASSES <= 1:
        raise ValueError("NUM_CLASSES must be > 1")

    if not 0 < VAL_SIZE < 1:
        raise ValueError("VAL_SIZE must be between 0 and 1")

    # check paths exist
    if not DATA_ROOT.exists():
        raise FileNotFoundError(f"DATA_ROOT not found: {DATA_ROOT}")

    if not TRAIN_IMAGE_DIR.exists():
        raise FileNotFoundError(f"TRAIN_IMAGE_DIR not found: {TRAIN_IMAGE_DIR}")

    # check both CSVs
    if not TRAIN_CSV_PATH.exists():
        raise FileNotFoundError(f"TRAIN_CSV_PATH not found: {TRAIN_CSV_PATH}")

    if not TEST_CSV_PATH.exists():
        raise FileNotFoundError(f"TEST_CSV_PATH not found: {TEST_CSV_PATH}")

    # check both image folders
    if not TEST_IMAGE_DIR.exists():
        raise FileNotFoundError(f"TEST_IMAGE_DIR not found: {TEST_IMAGE_DIR}")

In [8]:
# ============================================================
# PRINT CONFIG
# ============================================================

def print_config():

    print("=" * 60)
    print("EXPERIMENT CONFIG")
    print("=" * 60)

    print("\nDATA PATHS:")
    print(f"  DATA_ROOT: {DATA_ROOT}")
    print(f"  TRAIN_CSV_PATH: {TRAIN_CSV_PATH}")
    print(f"  TEST_CSV_PATH: {TEST_CSV_PATH}")
    print(f"  TEST_IMAGE_DIR: {TEST_IMAGE_DIR}")

    print("\nDATASET:")
    print(f"  IMAGE_COL: {IMAGE_COL}")
    print(f"  LABEL_COL: {LABEL_COL}")
    print(f"  NUM_CLASSES: {NUM_CLASSES}")

    print("\nDEVICE:")
    print(f"  DEVICE_MODE: {DEVICE_MODE}")
    print(f"  DEVICE: {DEVICE}")

    print("\nIMAGE:")
    print(f"  IMAGE_SIZE: {IMAGE_SIZE}")

    print("\nTRAINING:")
    print(f"  BATCH_SIZE: {BATCH_SIZE}")
    print(f"  NUM_WORKERS: {NUM_WORKERS}")
    print(f"  SEED: {SEED}")

    print("\nAUGMENTATION:")
    print(f"  USE_AUGMENTATION: {USE_AUGMENTATION}")
    print(f"  FLIP: {USE_RANDOM_FLIP}")
    print(f"  ROTATION: {USE_RANDOM_ROTATION}")
    print(f"  COLOR_JITTER: {USE_COLOR_JITTER}")
    print(f"  RANDOM_CROP: {USE_RANDOM_RESIZED_CROP}")

    print("\nIMBALANCE HANDLING:")
    print(f"  USE_WEIGHTED_LOSS: {USE_WEIGHTED_LOSS}")
    print(f"  USE_WEIGHTED_SAMPLER: {USE_WEIGHTED_SAMPLER}")

    print("\nLOSS:")
    print(f"  LOSS_NAME: {LOSS_NAME}")
    print(f"  FOCAL_GAMMA: {FOCAL_GAMMA}")

    print("\nOPTIMIZER:")
    print(f"  OPTIMIZER_NAME: {OPTIMIZER_NAME}")
    print(f"  LEARNING_RATE: {LEARNING_RATE}")
    print(f"  WEIGHT_DECAY: {WEIGHT_DECAY}")
    print(f"  MOMENTUM: {MOMENTUM}")

    print("\nVALIDATION:")
    print(f"  VAL_SIZE: {VAL_SIZE}")
    print(f"  STRATIFIED: {USE_STRATIFIED_SPLIT}")

    print("=" * 60)


# ============================================================
# RUN CHECKS
# ============================================================

validate_config()
print_config()

EXPERIMENT CONFIG

DATA PATHS:
  DATA_ROOT: /kaggle/input/datasets/paulconrad21/eye-disease-dataset/eye-disease-data
  TRAIN_CSV_PATH: /kaggle/input/datasets/paulconrad21/eye-disease-dataset/eye-disease-data/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv
  TEST_CSV_PATH: /kaggle/input/datasets/paulconrad21/eye-disease-dataset/eye-disease-data/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv
  TEST_IMAGE_DIR: /kaggle/input/datasets/paulconrad21/eye-disease-dataset/eye-disease-data/1. Original Images/b. Testing Set

DATASET:
  IMAGE_COL: image
  LABEL_COL: level
  NUM_CLASSES: 5

DEVICE:
  DEVICE_MODE: auto
  DEVICE: cpu

IMAGE:
  IMAGE_SIZE: 224

TRAINING:
  BATCH_SIZE: 16
  NUM_WORKERS: 2
  SEED: 42

AUGMENTATION:
  USE_AUGMENTATION: False
  FLIP: False
  ROTATION: False
  COLOR_JITTER: False
  RANDOM_CROP: False

IMBALANCE HANDLING:
  USE_WEIGHTED_LOSS: True
  USE_WEIGHTED_SAMPLER: False

LOSS:
  LOSS_NAME: weighted_cross_entropy
  FOCAL_GAMMA: 2.1288588684870

In [9]:
# ============================================================
# Load CSV files
# ============================================================

train_full_df = pd.read_csv(TRAIN_CSV_PATH)
test_df = pd.read_csv(TEST_CSV_PATH)

print("Training CSV columns:")
print(train_full_df.columns.tolist())

print("\nTesting CSV columns:")
print(test_df.columns.tolist())

print("\nTraining CSV preview:")
display(train_full_df.head())

print("\nTesting CSV preview:")
display(test_df.head())


Training CSV columns:
['Image name', 'Retinopathy grade', 'Risk of macular edema ', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']

Testing CSV columns:
['Image name', 'Retinopathy grade', 'Risk of macular edema ']

Training CSV preview:


,Image name,Retinopathy grade,Risk of macular edema,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,IDRiD_001,3,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IDRiD_002,3,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IDRiD_003,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IDRiD_004,3,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IDRiD_005,4,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Testing CSV preview:


,Image name,Retinopathy grade,Risk of macular edema
0,IDRiD_001,4,0
1,IDRiD_002,4,1
2,IDRiD_003,4,0
3,IDRiD_004,4,0
4,IDRiD_005,4,1


In [10]:
# ============================================================
# SELECT COLUMNS
# ============================================================
IMAGE_COL = "Image name"
LABEL_COL = "Retinopathy grade"

# Keep only the columns needed for this classification task
train_full_df = train_full_df[[IMAGE_COL, LABEL_COL]].copy()
test_df = test_df[[IMAGE_COL, LABEL_COL]].copy()

# Make sure labels are integers
train_full_df[LABEL_COL] = train_full_df[LABEL_COL].astype(int)
test_df[LABEL_COL] = test_df[LABEL_COL].astype(int)

print("Using image column:", IMAGE_COL)
print("Using label column:", LABEL_COL)

print("\nTrain class distribution:")
print(train_full_df[LABEL_COL].value_counts().sort_index())

print("\nTest class distribution:")
print(test_df[LABEL_COL].value_counts().sort_index())

Using image column: Image name
Using label column: Retinopathy grade

Train class distribution:
Retinopathy grade
0    134
1     20
2    136
3     74
4     49
Name: count, dtype: int64

Test class distribution:
Retinopathy grade
0    34
1     5
2    32
3    19
4    13
Name: count, dtype: int64


In [11]:
# ============================================================
# Dataset class
# ============================================================

class FundusDataset(Dataset):
    """
    Dataset for loading fundus images and labels.

    Input:
        dataframe: table with image names and labels
        image_dir: folder containing the images
        transform: preprocessing / augmentation
        image_col: column containing image names
        label_col: column containing labels

    Output per item:
        image tensor, label
    """

    def __init__(self, dataframe, image_dir, transform=None, image_col=None, label_col=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.image_col = image_col
        self.label_col = label_col

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_name = str(row[self.image_col])

        # Add .jpg if the CSV only contains IDRiD_001 instead of IDRiD_001.jpg
        if not image_name.lower().endswith((".jpg", ".jpeg", ".png")):
            image_name += ".jpg"

        image_path = self.image_dir / image_name

        image = Image.open(image_path).convert("RGB")
        label = int(row[self.label_col])

        if self.transform is not None:
            image = self.transform(image)

        return image, label

In [12]:
# ============================================================
# Class weights and weighted sampler
# ============================================================

def compute_class_weights(labels, num_classes):
    """
    Computes class weights for weighted loss.

    Rare classes get higher weights.
    """

    labels = np.array(labels)
    class_counts = np.bincount(labels, minlength=num_classes)

    # Avoid division by zero
    class_counts = np.maximum(class_counts, 1)

    total_samples = len(labels)
    class_weights = total_samples / (num_classes * class_counts)

    return torch.tensor(class_weights, dtype=torch.float32)


def create_weighted_sampler(labels):
    """
    Creates a WeightedRandomSampler.

    Rare classes are sampled more often during training.
    """

    labels = np.array(labels)
    class_counts = np.bincount(labels)

    class_counts = np.maximum(class_counts, 1)

    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels]

    sampler = WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.float32),
        num_samples=len(sample_weights),
        replacement=True
    )

    return sampler

In [13]:
# ============================================================
# Preprocessing Options for Fundus Images
# Options:
#   "standard"
#   "green_channel"
#   "clahe"
#   "gamma_correction"
#   "gray_gamma_clahe"                          # RGB -> gray -> gamma adjustment -> CLAHE
#   "green_histmatch_median_clahe_unsharp_crop" # green -> histogram match -> median -> CLAHE -> unsharp -> remove black corners
#   "green_clahe_bgsub_mask"                    # green -> CLAHE -> background subtraction -> FOV mask
#   "ben_graham"
#   "white_balance"
#   "white_balance_clahe"
# ============================================================
PREPROCESSING_MODES = [
    "standard",
    "green_channel",
    "clahe",
    "clean_green_masked",
    "gamma_correction",
    "gray_gamma_clahe",
    "green_histmatch_median_clahe_unsharp_crop",
    "green_clahe_bgsub_mask",
    "ben_graham",
    "white_balance",
    "white_balance_clahe",
]


class FundusPreprocessing:
    def __init__(self, mode="standard", gamma=1.2, reference_image_path=None):
        self.mode = mode
        self.gamma = gamma
        self.reference_green = None

        if reference_image_path is not None:
            try:
                ref_img = Image.open(reference_image_path).convert("RGB")
                ref_img = np.array(ref_img)
                self.reference_green = ref_img[:, :, 1]
            except Exception as error:
                print(f"Warning: could not load reference image for histogram matching: {error}")
                self.reference_green = None

    def _apply_gamma_rgb(self, img, gamma=None):
        """Gamma correction on RGB uint8 image."""
        gamma = self.gamma if gamma is None else gamma
        inv_gamma = 1.0 / gamma
        table = np.array([
            ((i / 255.0) ** inv_gamma) * 255
            for i in np.arange(256)
        ]).astype("uint8")
        return cv2.LUT(img, table)

    def _apply_clahe_gray(self, gray, clip_limit=1.5, tile_grid_size=(8, 8)):
        """CLAHE on a single-channel grayscale image."""
        clahe = cv2.createCLAHE(
            clipLimit=clip_limit,
            tileGridSize=tile_grid_size,
        )
        return clahe.apply(gray)

    def _apply_clahe_lab(self, img, clip_limit=1.5, tile_grid_size=(8, 8)):
        """CLAHE on L channel in LAB color space."""
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(
            clipLimit=clip_limit,
            tileGridSize=tile_grid_size,
        )
        l = clahe.apply(l)
        lab = cv2.merge((l, a, b))
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    def _apply_white_balance(self, img):
        """Gray-world white balance."""
        img = img.astype(np.float32)
        r_avg = max(np.mean(img[:, :, 0]), 1e-6)
        g_avg = max(np.mean(img[:, :, 1]), 1e-6)
        b_avg = max(np.mean(img[:, :, 2]), 1e-6)
        gray = (r_avg + g_avg + b_avg) / 3.0
        img[:, :, 0] *= gray / r_avg
        img[:, :, 1] *= gray / g_avg
        img[:, :, 2] *= gray / b_avg
        return np.clip(img, 0, 255).astype(np.uint8)

    def _histogram_match_gray(self, gray):
        """Histogram match gray channel to a reference green channel, fallback to equalizeHist."""
        if match_histograms is not None and self.reference_green is not None:
            matched = match_histograms(gray, self.reference_green, channel_axis=None)
            return np.clip(matched, 0, 255).astype(np.uint8)
        return cv2.equalizeHist(gray)

    def _unsharp_gray(self, gray, sigma=1.0, amount=1.5):
        """Unsharp masking for a grayscale image."""
        blurred = cv2.GaussianBlur(gray, (0, 0), sigma)
        sharpened = cv2.addWeighted(gray, 1.0 + amount, blurred, -amount, 0)
        return np.clip(sharpened, 0, 255).astype(np.uint8)

    def _remove_black_corners_gray(self, gray, threshold=8):
        """Crop the bounding box around non-black pixels in a grayscale image."""
        mask = gray > threshold
        coords = np.argwhere(mask)

        if coords.size == 0:
            return gray

        y0, x0 = coords.min(axis=0)
        y1, x1 = coords.max(axis=0) + 1
        return gray[y0:y1, x0:x1]

    def _green_clahe_bgsub_mask(self, img, threshold=10, sigma=15):
        """
        Pipeline:
        RGB -> BGR -> FOV mask -> green channel -> CLAHE -> background subtraction
        -> apply FOV mask -> 3-channel grayscale output.

        sigma=15 is chosen for 224x224 training input. If you visualize too much
        structure removal, try sigma=10. If illumination is still uneven, try sigma=20.
        """
        img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        # FOV mask from non-black retinal field.
        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)

        # Green channel carries strong vessel/lesion contrast in fundus images.
        green = img_bgr[:, :, 1]

        # Local contrast enhancement.
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced_green = clahe.apply(green)

        # Background subtraction / illumination flattening.
        bg = cv2.GaussianBlur(enhanced_green, (0, 0), sigmaX=sigma)
        equalized = cv2.addWeighted(enhanced_green, 4, bg, -4, 128)

        # Remove boundary halo/artifacts by applying the retina mask.
        final_cleaned = cv2.bitwise_and(equalized, equalized, mask=mask)

        # Convert single-channel result to 3 channels for ImageNet-pretrained ViT.
        final_rgb = np.stack([final_cleaned, final_cleaned, final_cleaned], axis=-1)
        return final_rgb

    def __call__(self, img):
        img = np.array(img.convert("RGB"))

        if self.mode == "standard":
            return Image.fromarray(img)

        elif self.mode == "green_channel":
            green = img[:, :, 1]
            img = np.stack([green, green, green], axis=-1)
            return Image.fromarray(img)

        elif self.mode == "clahe":
            img = self._apply_clahe_lab(img, clip_limit=1.0, tile_grid_size=(8, 8))
            return Image.fromarray(img)

        elif self.mode == "gamma_correction":
            img = self._apply_gamma_rgb(img, gamma=self.gamma)
            return Image.fromarray(img)

        elif self.mode == "gray_gamma_clahe":
            # Correct order: RGB -> gray -> gamma adjustment -> CLAHE -> 3-channel grayscale image.
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
            gray_rgb = np.stack([gray, gray, gray], axis=-1)
            gamma_img = self._apply_gamma_rgb(gray_rgb, gamma=self.gamma)
            gamma_gray = gamma_img[:, :, 0]
            clahe_gray = self._apply_clahe_gray(gamma_gray, clip_limit=1.5, tile_grid_size=(8, 8))
            img = np.stack([clahe_gray, clahe_gray, clahe_gray], axis=-1)
            return Image.fromarray(img)

        elif self.mode == "green_histmatch_median_clahe_unsharp_crop":
            # Correct order: green -> histogram match -> median -> CLAHE -> unsharp -> remove black corners.
            green = img[:, :, 1]
            green = self._histogram_match_gray(green)
            green = cv2.medianBlur(green, ksize=3)
            green = self._apply_clahe_gray(green, clip_limit=1.5, tile_grid_size=(8, 8))
            green = self._unsharp_gray(green, sigma=1.0, amount=1.5)
            green = self._remove_black_corners_gray(green, threshold=8)
            img = np.stack([green, green, green], axis=-1)
            return Image.fromarray(img)

        elif self.mode == "green_clahe_bgsub_mask":
            img = self._green_clahe_bgsub_mask(img, threshold=10, sigma=15)
            return Image.fromarray(img)

        elif self.mode == "ben_graham":
            img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            img_bgr = cv2.addWeighted(
                img_bgr,
                4,
                cv2.GaussianBlur(img_bgr, (0, 0), 10),
                -4,
                128,
            )
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            return Image.fromarray(img_rgb)

        elif self.mode == "clean_green_masked":
                    """
                    Pipeline:
                    RGB -> grayscale FOV mask -> green channel -> CLAHE
                    -> background subtraction -> apply FOV mask
                    -> 3-channel grayscale output.
                    """
        
                    # FOV mask from original RGB image
                    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        
                    _, mask = cv2.threshold(
                        gray,
                        10,
                        255,
                        cv2.THRESH_BINARY
                    )
        
                    # Green channel
                    green_ch = img[:, :, 1]
        
                    # CLAHE enhancement
                    clahe = cv2.createCLAHE(
                        clipLimit=2.0,
                        tileGridSize=(8, 8)
                    )
        
                    enhanced_green = clahe.apply(green_ch)
        
                    # Background subtraction / illumination normalization
                    bg = cv2.GaussianBlur(
                        enhanced_green,
                        (0, 0),
                        sigmaX=30
                    )
        
                    equalized = cv2.addWeighted(
                        enhanced_green,
                        4,
                        bg,
                        -4,
                        128
                    )
        
                    # Apply FOV mask to remove border halos
                    final_cleaned = cv2.bitwise_and(
                        equalized,
                        equalized,
                        mask=mask
                    )
        
                    # Convert single channel to 3 channels for ViT/ImageNet normalization
                    final_rgb = np.stack(
                        [final_cleaned, final_cleaned, final_cleaned],
                        axis=-1
                    )
        
                    return Image.fromarray(final_rgb)

        elif self.mode == "white_balance":
            img = self._apply_white_balance(img)
            return Image.fromarray(img)

        elif self.mode == "white_balance_clahe":
            img = self._apply_white_balance(img)
            img = self._apply_clahe_lab(img, clip_limit=1.5, tile_grid_size=(8, 8))
            return Image.fromarray(img)

        else:
            raise ValueError(
                f"Unknown preprocessing_mode: {self.mode}. Use one of: {PREPROCESSING_MODES}"
            )


In [14]:
# ============================================================
# Augmented Dataset and DataLoaders
# ============================================================
class AugmentedDataset(Dataset):
    """Repeats the base dataset N times. Each access applies random augmentation again."""

    def __init__(self, base_dataset, multiplier=1):
        self.base_dataset = base_dataset
        self.multiplier = multiplier

    def __len__(self):
        return len(self.base_dataset) * self.multiplier

    def __getitem__(self, idx):
        real_idx = idx % len(self.base_dataset)
        return self.base_dataset[real_idx]


def create_dataloaders(
    BATCH_SIZE,
    data_augmentation=1,
    preprocessing_mode="standard",
):
    """
    Creates train/validation/test loaders with selectable fundus preprocessing.

    Important:
    - No class-specific oversampling multipliers are used here.
    - The training split keeps its original class distribution.
    - data_augmentation only repeats the whole train dataset uniformly.
    """

    if USE_STRATIFIED_SPLIT:
        train_df, val_df = train_test_split(
            train_full_df,
            test_size=VAL_SIZE,
            random_state=SEED,
            stratify=train_full_df[LABEL_COL],
        )
    else:
        train_df, val_df = train_test_split(
            train_full_df,
            test_size=VAL_SIZE,
            random_state=SEED,
        )

    # Use one fixed training image as the reference for histogram matching.
    # This keeps the histogram-matching pipeline deterministic and prevents leakage from validation/test.
    reference_image_path = _resolve_image_path(
        TRAIN_IMAGE_DIR,
        train_df.iloc[0][IMAGE_COL],
    )

    train_transform = transforms.Compose([
        FundusPreprocessing(preprocessing_mode, reference_image_path=reference_image_path),
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(
            brightness=0.15,
            contrast=0.15,
            saturation=0.10,
            hue=0.02,
        ),
        transforms.RandomAdjustSharpness(sharpness_factor=2.0, p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    eval_transform = transforms.Compose([
        FundusPreprocessing(preprocessing_mode, reference_image_path=reference_image_path),
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    base_train_dataset = FundusDataset(
        dataframe=train_df,
        image_dir=TRAIN_IMAGE_DIR,
        transform=train_transform,
        image_col=IMAGE_COL,
        label_col=LABEL_COL,
    )

    train_dataset = AugmentedDataset(
        base_dataset=base_train_dataset,
        multiplier=data_augmentation,
    )

    val_dataset = FundusDataset(
        dataframe=val_df,
        image_dir=TRAIN_IMAGE_DIR,
        transform=eval_transform,
        image_col=IMAGE_COL,
        label_col=LABEL_COL,
    )

    test_dataset = FundusDataset(
        dataframe=test_df,
        image_dir=TEST_IMAGE_DIR,
        transform=eval_transform,
        image_col=IMAGE_COL,
        label_col=LABEL_COL,
    )

    if USE_WEIGHTED_LOSS:
        class_weights = compute_class_weights(
            labels=train_df[LABEL_COL].values,
            num_classes=NUM_CLASSES,
        )
    else:
        class_weights = None

    if USE_WEIGHTED_SAMPLER:
        labels_for_sampler = np.tile(
            train_df[LABEL_COL].values,
            data_augmentation,
        )
        train_sampler = create_weighted_sampler(labels_for_sampler)
        shuffle_train = False
    else:
        train_sampler = None
        shuffle_train = True

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle_train,
        sampler=train_sampler,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    return (
        train_loader,
        val_loader,
        test_loader,
        class_weights,
        train_df,
        val_df,
    )


In [15]:
# ============================================================
# Variable depth MLP 
# ============================================================
def create_mlp_head(
    in_features,
    num_classes,
    mlp_depth,
    hidden_dim,
    dropout
):

    layers = []

    current_dim = in_features

    for _ in range(mlp_depth):

        layers.append(nn.Linear(current_dim, hidden_dim))
        layers.append(nn.GELU())
        layers.append(nn.Dropout(dropout))

        current_dim = hidden_dim

    layers.append(nn.Linear(current_dim, num_classes))

    return nn.Sequential(*layers)

In [16]:
# ============================================================
# Vision Transformer (ViT)
# ============================================================
def create_model(
    num_classes,
    dropout,
    mlp_depth,
    hidden_dim
):

    weights = ViT_B_16_Weights.IMAGENET1K_V1

    model = models.vit_b_16(weights=weights)

    in_features = model.heads.head.in_features

    model.heads.head = create_mlp_head(
        in_features=in_features,
        num_classes=num_classes,
        mlp_depth=mlp_depth,
        hidden_dim=hidden_dim,
        dropout=dropout
    )

    for param in model.parameters():
        param.requires_grad = True

    return model

In [17]:
# ============================================================
# Loss function
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = nn.functional.cross_entropy(
            logits,
            targets,
            weight=self.alpha,
            reduction="none"
        )
        pt = torch.exp(-ce_loss)
        loss = ((1 - pt) ** self.gamma) * ce_loss
        return loss.mean()


# ============================================================
# Create Loss Function
# ============================================================

def create_loss(
    LOSS_NAME,
    class_weights,
    DEVICE,
    FOCAL_GAMMA
):

    if LOSS_NAME == "cross_entropy":

        return nn.CrossEntropyLoss()

    elif LOSS_NAME == "weighted_cross_entropy":

        return nn.CrossEntropyLoss(
            weight=class_weights.to(DEVICE)
        )

    elif LOSS_NAME == "focal_loss":

        return FocalLoss(
            gamma=FOCAL_GAMMA
        )

    elif LOSS_NAME == "weighted_focal_loss":

        return FocalLoss(
            alpha=class_weights.to(DEVICE),
            gamma=FOCAL_GAMMA
        )

    else:
        raise ValueError("Invalid LOSS_NAME")

In [18]:
# ============================================================
# Train one epoch
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion):

    model.train()

    running_loss = 0.0

    for images, labels in tqdm(loader, desc="Training", leave=False):

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        logits = model(images)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(loader.dataset)

    return epoch_loss

In [19]:
# ============================================================
# Metrics and evaluation helpers
# ============================================================
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "kappa": cohen_kappa_score(y_true, y_pred, weights="quadratic"),
    }


def evaluate(model, loader, criterion):
    model.eval()
    all_preds = []
    all_labels = []
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validation", leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(images)
            loss = criterion(logits, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(all_labels, all_preds)

    return epoch_loss, metrics


def evaluate_detailed(model, loader, criterion):
    model.eval()
    all_preds = []
    all_labels = []
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluating", leave=False):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = model(images)
            loss = criterion(logits, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(all_labels, all_preds)

    return epoch_loss, metrics, np.array(all_preds), np.array(all_labels)


In [20]:
# ============================================================
# Visualization helpers
# ============================================================
def plot_training_curves(history, title_prefix=""):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix} Training and Validation Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, history["val_macro_f1"], label="Validation Macro F1")
    plt.plot(epochs, history["val_macro_accuracy"], label="Validation Macro Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title(f"{title_prefix} Validation Scores")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


def plot_confusion_matrix_for_experiment(y_true, y_pred, num_classes, title):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))

    fig, ax = plt.subplots(figsize=(10, 10))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=np.arange(num_classes),
    )
    disp.plot(
        cmap="Blues",
        ax=ax,
        xticks_rotation=45,
        colorbar=True,
    )
    plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_confidence_by_class(model, test_loader, experiment_name):
    model.eval()

    all_preds = []
    all_labels = []
    all_confidences = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)

            logits = model(images)
            probs = F.softmax(logits, dim=1)
            confidences, preds = torch.max(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_confidences.extend(confidences.cpu().numpy())

    results_df = pd.DataFrame({
        "true_label": all_labels,
        "predicted_label": all_preds,
        "confidence": all_confidences,
    })

    results_df["correct"] = (
        results_df["true_label"] == results_df["predicted_label"]
    )

    correct_conf = (
        results_df[results_df["correct"]]
        .groupby("true_label")["confidence"]
        .mean()
    )

    wrong_conf = (
        results_df[~results_df["correct"]]
        .groupby("true_label")["confidence"]
        .mean()
    )

    classes = sorted(results_df["true_label"].unique())

    correct_values = [correct_conf.get(c, 0) for c in classes]
    wrong_values = [wrong_conf.get(c, 0) for c in classes]

    x = np.arange(len(classes))
    width = 0.35

    plt.figure(figsize=(10, 6))
    plt.bar(x - width / 2, correct_values, width, label="Correct")
    plt.bar(x + width / 2, wrong_values, width, label="Incorrect")
    plt.xlabel("True Class")
    plt.ylabel("Average Confidence")
    plt.title(f"{experiment_name}: Confidence by Class — Correct vs Incorrect")
    plt.xticks(x, classes)
    plt.ylim(0, 1)
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    plt.show()

    return results_df




def _resolve_image_path(image_dir, image_name):
    """Resolve IDRiD image filenames with or without extension."""
    image_name = str(image_name)
    image_dir = Path(image_dir)

    candidate = image_dir / image_name
    if candidate.exists():
        return candidate

    for ext in [".jpg", ".jpeg", ".png", ".tif", ".tiff"]:
        candidate = image_dir / f"{image_name}{ext}"
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Could not find image: {image_name} in {image_dir}")


def plot_preprocessing_examples_by_class(
    dataframe,
    image_dir,
    preprocessing_mode,
    image_col=IMAGE_COL,
    label_col=LABEL_COL,
    num_classes=NUM_CLASSES,
    seed=SEED,
):
    """
    Shows one random training image per class before and after the selected preprocessing.

    Layout:
        Row = class
        Column 1 = original image
        Column 2 = preprocessed image
    """
    rng = np.random.default_rng(seed)
    reference_image_path = _resolve_image_path(
        image_dir,
        dataframe.iloc[0][image_col],
    )
    preprocessor = FundusPreprocessing(
        preprocessing_mode,
        reference_image_path=reference_image_path,
    )

    selected_rows = []
    for class_id in range(num_classes):
        class_df = dataframe[dataframe[label_col] == class_id]
        if len(class_df) == 0:
            continue
        row_idx = rng.choice(class_df.index.to_numpy())
        selected_rows.append(dataframe.loc[row_idx])

    if len(selected_rows) == 0:
        print("No images available for preprocessing visualization.")
        return

    fig, axes = plt.subplots(
        nrows=len(selected_rows),
        ncols=2,
        figsize=(8, 4 * len(selected_rows)),
    )

    if len(selected_rows) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_position, row in enumerate(selected_rows):
        image_path = _resolve_image_path(image_dir, row[image_col])
        original = Image.open(image_path).convert("RGB")
        processed = preprocessor(original)

        axes[row_position, 0].imshow(original)
        axes[row_position, 0].set_title(f"Original - Class {int(row[label_col])}")
        axes[row_position, 0].axis("off")

        axes[row_position, 1].imshow(processed, cmap="gray")
        axes[row_position, 1].set_title(f"{preprocessing_mode} - Class {int(row[label_col])}")
        axes[row_position, 1].axis("off")

    plt.suptitle(f"Preprocessing effect on random training images: {preprocessing_mode}", y=1.01)
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# Extract ViT CLS Embeddings + t-SNE visualization
# ============================================================
def extract_vit_cls_embeddings(model, loader, device):
    """Extracts CLS-token embeddings after the ViT encoder and before the classification head."""

    model.eval()

    all_embeddings = []
    all_labels = []
    all_preds = []
    all_confidences = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            x = model._process_input(images)

            batch_size = x.shape[0]
            cls_token = model.class_token.expand(batch_size, -1, -1)

            x = torch.cat((cls_token, x), dim=1)
            x = model.encoder(x)

            embeddings = x[:, 0]
            logits = model.heads(embeddings)
            probs = torch.softmax(logits, dim=1)

            confidences, preds = torch.max(probs, dim=1)

            all_embeddings.append(embeddings.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_confidences.extend(confidences.cpu().numpy())

    all_embeddings = np.concatenate(all_embeddings, axis=0)

    return (
        all_embeddings,
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_confidences),
    )


def plot_tsne_embeddings(model, test_loader, experiment_name):
    test_embeddings, test_labels, test_preds, test_confidences = extract_vit_cls_embeddings(
        model=model,
        loader=test_loader,
        device=DEVICE,
    )

    print("Embedding shape:", test_embeddings.shape)
    print("Labels shape:", test_labels.shape)
    print("Predictions shape:", test_preds.shape)

    tsne = TSNE(
        n_components=2,
        perplexity=10,
        learning_rate="auto",
        init="pca",
        random_state=SEED,
    )

    embeddings_tsne = tsne.fit_transform(test_embeddings)

    cmap = ListedColormap([
        "blue",
        "orange",
        "green",
        "red",
        "purple",
    ])

    plt.figure(figsize=(9, 7))
    scatter = plt.scatter(
        embeddings_tsne[:, 0],
        embeddings_tsne[:, 1],
        c=test_labels,
        cmap=cmap,
        vmin=0,
        vmax=NUM_CLASSES - 1,
        s=70,
        alpha=0.85,
    )

    cbar = plt.colorbar(scatter, ticks=list(range(NUM_CLASSES)))
    cbar.set_label("True Class")

    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.title(f"{experiment_name}: ViT CLS Embeddings — t-SNE Projection")
    plt.grid(alpha=0.3)
    plt.show()

    return test_embeddings, test_labels, test_preds, test_confidences


In [ ]:
# ============================================================
# Single preprocessing experiment runner
# ============================================================
def run_preprocessing_experiment(preprocessing_mode):
    print("=" * 70)
    print(f"PREPROCESSING EXPERIMENT: {preprocessing_mode}")
    print("=" * 70)

    set_seed(SEED)

    train_loader, val_loader, test_loader, class_weights, train_df, val_df = create_dataloaders(
        BATCH_SIZE=BATCH_SIZE,
        data_augmentation=DATA_AUGMENTATION_MULTIPLIER,
        preprocessing_mode=preprocessing_mode,
    )

    print(f"Preprocessing mode: {preprocessing_mode}")
    print(f"Train samples: {len(train_df)}")
    print("No class-specific oversampling multipliers are used.")
    print(f"Augmented train dataset size: {len(train_loader.dataset)}")
    print(f"Validation samples: {len(val_df)}")
    print(f"Test samples: {len(test_df)}")

    if class_weights is not None:
        print("Class weights:", class_weights)

    # Visual check: one random training image per class before/after preprocessing.
    plot_preprocessing_examples_by_class(
        dataframe=train_df,
        image_dir=TRAIN_IMAGE_DIR,
        preprocessing_mode=preprocessing_mode,
        image_col=IMAGE_COL,
        label_col=LABEL_COL,
        num_classes=NUM_CLASSES,
        seed=SEED,
    )

    model = create_model(
        num_classes=NUM_CLASSES,
        dropout=DROPOUT,
        mlp_depth=MLP_DEPTH,
        hidden_dim=HIDDEN_DIM,
    ).to(DEVICE)

    criterion = create_loss(
        LOSS_NAME=LOSS_NAME,
        class_weights=class_weights,
        DEVICE=DEVICE,
        FOCAL_GAMMA=FOCAL_GAMMA,
    )

    if OPTIMIZER_NAME == "adam":
        optimizer = optim.Adam(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )
    elif OPTIMIZER_NAME == "adamw":
        optimizer = optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )
    elif OPTIMIZER_NAME == "sgd":
        optimizer = optim.SGD(
            model.parameters(),
            lr=LEARNING_RATE,
            momentum=MOMENTUM,
            weight_decay=WEIGHT_DECAY,
        )
    else:
        raise ValueError(f"Unknown optimizer: {OPTIMIZER_NAME}")

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_accuracy": [],
        "val_macro_accuracy": [],
        "val_macro_f1": [],
        "val_kappa": [],
    }

    best_val_score = -np.inf
    best_epoch = -1
    best_model_state = None
    epochs_without_improvement = 0

    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_metrics = evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_metrics["accuracy"])
        history["val_macro_accuracy"].append(val_metrics["macro_accuracy"])
        history["val_macro_f1"].append(val_metrics["macro_f1"])
        history["val_kappa"].append(val_metrics["kappa"])

        current_score = val_metrics["macro_f1"]

        print(f"Train loss: {train_loss:.4f}")
        print(f"Val loss: {val_loss:.4f}")
        print(f"Val accuracy: {val_metrics['accuracy']:.4f}")
        print(f"Val macro accuracy: {val_metrics['macro_accuracy']:.4f}")
        print(f"Val macro F1: {val_metrics['macro_f1']:.4f}")
        print(f"Val kappa: {val_metrics['kappa']:.4f}")

        if current_score > best_val_score:
            best_val_score = current_score
            best_epoch = epoch
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
            torch.save(
                best_model_state,
                f"/kaggle/working/best_model_{preprocessing_mode}.pth",
            )
            print("New best model saved.")
        else:
            epochs_without_improvement += 1
            print(f"No improvement: {epochs_without_improvement}/{PATIENCE}")

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_model_state)

    plot_training_curves(history, title_prefix=preprocessing_mode)

    test_loss, test_metrics, test_preds, test_labels = evaluate_detailed(
        model,
        test_loader,
        criterion,
    )

    print("\n" + "=" * 70)
    print(f"TEST RESULTS: {preprocessing_mode}")
    print("=" * 70)
    print(f"Best validation macro F1: {best_val_score:.6f}")
    print(f"Best epoch: {best_epoch}")
    print(f"Test loss: {test_loss:.4f}")
    print(f"Test accuracy: {test_metrics['accuracy']:.4f}")
    print(f"Test macro accuracy: {test_metrics['macro_accuracy']:.4f}")
    print(f"Test macro F1: {test_metrics['macro_f1']:.4f}")
    print(f"Test kappa: {test_metrics['kappa']:.4f}")

    print("\nClassification report:")
    print(classification_report(test_labels, test_preds, digits=4))

    plot_confusion_matrix_for_experiment(
        y_true=test_labels,
        y_pred=test_preds,
        num_classes=NUM_CLASSES,
        title=f"{preprocessing_mode}: Confusion Matrix - Test Set",
    )

    confidence_df = plot_confidence_by_class(
        model=model,
        test_loader=test_loader,
        experiment_name=preprocessing_mode,
    )

    test_embeddings, emb_labels, emb_preds, emb_confidences = plot_tsne_embeddings(
        model=model,
        test_loader=test_loader,
        experiment_name=preprocessing_mode,
    )

    return {
        "preprocessing_mode": preprocessing_mode,
        "model": model,
        "history": history,
        "best_val_macro_f1": best_val_score,
        "best_epoch": best_epoch,
        "test_loss": test_loss,
        "test_metrics": test_metrics,
        "test_labels": test_labels,
        "test_preds": test_preds,
        "confidence_df": confidence_df,
        "test_embeddings": test_embeddings,
        "embedding_labels": emb_labels,
        "embedding_preds": emb_preds,
        "embedding_confidences": emb_confidences,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
    }


# Experiment: standard

In [ ]:
# standard_results = run_preprocessing_experiment("standard")

# Experiment: green_channel

In [ ]:
# green_channel_results = run_preprocessing_experiment("green_channel")

# Experiment: clahe

In [ ]:
# clahe_results = run_preprocessing_experiment("clahe")

# Experiment: gamma_correction

In [ ]:
# gamma_correction_results = run_preprocessing_experiment("gamma_correction")

# Experiment: gray_gamma_clahe

In [ ]:
# gray_gamma_clahe_results = run_preprocessing_experiment("gray_gamma_clahe")

# Experiment: green_histmatch_median_clahe_unsharp_crop

In [ ]:
# green_histmatch_median_clahe_unsharp_crop_results = run_preprocessing_experiment("green_histmatch_median_clahe_unsharp_crop")

# Experiment: green_clahe_bgsub_mask

In [ ]:
green_clahe_bgsub_mask_results = run_preprocessing_experiment("green_clahe_bgsub_mask")

# Experiment: ben_graham

In [ ]:
ben_graham_results = run_preprocessing_experiment("clean_green_masked")

# Experiment: white_balance

In [ ]:
# white_balance_results = run_preprocessing_experiment("white_balance")

# Experiment: white_balance_clahe

In [ ]:
# white_balance_clahe_results = run_preprocessing_experiment("white_balance_clahe")

# Summary comparison

In [ ]:
# # ============================================================
# # Compare all finished preprocessing experiments
# # Run this after executing the individual experiment cells above.
# # ============================================================
# experiment_results = [
#     standard_results,
#     green_channel_results,
#     clahe_results,
#     gamma_correction_results,
#     gray_gamma_clahe_results,
#     green_histmatch_median_clahe_unsharp_crop_results,
#     green_clahe_bgsub_mask_results,
#     ben_graham_results,
#     white_balance_results,
#     white_balance_clahe_results,
# ]

# summary_df = pd.DataFrame([
#     {
#         "preprocessing_mode": r["preprocessing_mode"],
#         "best_val_macro_f1": r["best_val_macro_f1"],
#         "best_epoch": r["best_epoch"],
#         "test_accuracy": r["test_metrics"]["accuracy"],
#         "test_macro_accuracy": r["test_metrics"]["macro_accuracy"],
#         "test_macro_f1": r["test_metrics"]["macro_f1"],
#         "test_kappa": r["test_metrics"]["kappa"],
#     }
#     for r in experiment_results
# ])

# display(summary_df.sort_values("test_macro_f1", ascending=False))
